# Tarea 4: Modelo Recurrente (LSTM)

Este notebook contiene la preparación de secuencias temporales históricas y el entrenamiento de un modelo de Red Neuronal Recurrente con arquitectura LSTM en Keras/TensorFlow. Este modelo toma secuencias de los últimos 10 partidos para ambos equipos y predice de forma probabilística el resultado del encuentro actual, evitando estrictamente el data leakage temporal.

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, f1_score

PROCESSED_DIR = "../data/processed"
SAVED_MODELS_DIR = "../saved_models"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
clean_data_path = os.path.join(PROCESSED_DIR, "matches_clean.csv")

df = pd.read_csv(clean_data_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Cargados {len(df)} partidos para procesar en secuencias.")

### 1. Construcción de Secuencias Temporales

Para cada partido, extraemos la secuencia de los últimos 10 partidos jugados por el equipo local y el visitante antes del encuentro actual. 
Las features de cada paso de la secuencia son: `[resultado_perspectiva, goles_favor, goles_contra, elo_rival, es_local, tournament_weight]`.

In [ ]:
team_history = {}
seq_len = 10
n_features = 6

home_sequences = []
away_sequences = []
targets = []
years = []

for idx, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    res = row['result']
    home_g = row['home_score']
    away_g = row['away_score']
    elo_h = row['elo_home']
    elo_a = row['elo_away']
    weight = row['tournament_weight']
    
    # Secuencias Home
    hist_h = team_history.get(home, [])
    seq_h = hist_h[-seq_len:] if len(hist_h) >= seq_len else hist_h
    if len(seq_h) < seq_len:
        padding = [[0.0] * n_features] * (seq_len - len(seq_h))
        seq_h_padded = padding + seq_h
    else:
        seq_h_padded = seq_h
        
    # Secuencias Away
    hist_a = team_history.get(away, [])
    seq_a = hist_a[-seq_len:] if len(hist_a) >= seq_len else hist_a
    if len(seq_a) < seq_len:
        padding = [[0.0] * n_features] * (seq_len - len(seq_a))
        seq_a_padded = padding + seq_a
    else:
        seq_a_padded = seq_a
        
    home_sequences.append(seq_h_padded)
    away_sequences.append(seq_a_padded)
    targets.append(res)
    years.append(row['date'].year)
    
    # Actualizar históricos de forma cronológica
    if home not in team_history: team_history[home] = []
    team_history[home].append([res, home_g, away_g, elo_a, 1.0, weight])
    
    if away not in team_history: team_history[away] = []
    team_history[away].append([2 - res, away_g, home_g, elo_h, 0.0, weight])

X_home = np.array(home_sequences, dtype=np.float32)
X_away = np.array(away_sequences, dtype=np.float32)
y = np.array(targets, dtype=np.int32)
years = np.array(years)

print(f"Dimensiones de entrada: X_home {X_home.shape}, X_away {X_away.shape}")

### 2. Particionado de Fechas y Escalado de Secuencias

In [ ]:
train_mask = (years <= 2018)
val_mask = (years >= 2019) & (years <= 2021)
test_mask = (years >= 2022) & (years <= 2024)

X_home_train, X_away_train, y_train = X_home[train_mask], X_away[train_mask], y[train_mask]
X_home_val, X_away_val, y_val = X_home[val_mask], X_away[val_mask], y[val_mask]
X_home_test, X_away_test, y_test = X_home[test_mask], X_away[test_mask], y[test_mask]

# Escalado de las variables en la secuencia
scaler = StandardScaler()
train_combined = np.vstack([X_home_train.reshape(-1, n_features), X_away_train.reshape(-1, n_features)])
scaler.fit(train_combined)

def scale_sequences(X, scaler):
    n_samples = X.shape[0]
    X_reshaped = X.reshape(-1, n_features)
    X_scaled = scaler.transform(X_reshaped)
    return X_scaled.reshape(n_samples, seq_len, n_features)

X_home_train_scaled = scale_sequences(X_home_train, scaler)
X_away_train_scaled = scale_sequences(X_away_train, scaler)
X_home_val_scaled = scale_sequences(X_home_val, scaler)
X_away_val_scaled = scale_sequences(X_away_val, scaler)
X_home_test_scaled = scale_sequences(X_home_test, scaler)
X_away_test_scaled = scale_sequences(X_away_test, scaler)

print("Secuencias temporales divididas y escaladas correctamente.")

### 3. Definición y Entrenamiento de la Arquitectura LSTM

Definimos un modelo con dos ramas de entrada independientes (Home y Away) que comparten las capas recurrentes LSTM, reduciendo parámetros y favoreciendo una representación generalizable del rendimiento futbolístico.

In [ ]:
inputs_home = tf.keras.Input(shape=(seq_len, n_features), name='home_input')
inputs_away = tf.keras.Input(shape=(seq_len, n_features), name='away_input')

# Compartimos las capas LSTM entre ramas
lstm_1 = tf.keras.layers.LSTM(64, return_sequences=True, name='lstm_1')
dropout = tf.keras.layers.Dropout(0.3, name='dropout')
lstm_2 = tf.keras.layers.LSTM(32, name='lstm_2')

home_repr = lstm_2(dropout(lstm_1(inputs_home)))
away_repr = lstm_2(dropout(lstm_1(inputs_away)))

merged = tf.keras.layers.concatenate([home_repr, away_repr], name='merge_layer')
dense = tf.keras.layers.Dense(16, activation='relu', name='dense_layer')(merged)
outputs = tf.keras.layers.Dense(3, activation='softmax', name='output_layer')(dense)

model = tf.keras.Model(inputs=[inputs_home, inputs_away], outputs=outputs)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
]

model.summary()

In [ ]:
print("Iniciando entrenamiento del modelo...")
history = model.fit(
    x=[X_home_train_scaled, X_away_train_scaled],
    y=y_train,
    epochs=30,
    batch_size=64,
    validation_data=([X_home_val_scaled, X_away_val_scaled], y_val),
    callbacks=callbacks,
    verbose=1
)

### 4. Evaluación de Métricas y Guardado

In [ ]:
y_proba = model.predict([X_home_val_scaled, X_away_val_scaled])
y_pred = np.argmax(y_proba, axis=1)

acc = accuracy_score(y_val, y_pred)
loss = log_loss(y_val, y_proba)
f1_mac = f1_score(y_val, y_pred, average='macro')

print("\n--- Métricas del Modelo LSTM en Validación ---")
print(f"Accuracy: {acc:.6f}")
print(f"Log-loss: {loss:.6f}")
print(f"F1-macro: {f1_mac:.6f}")

# Guardado
model_path = os.path.join(SAVED_MODELS_DIR, "lstm_model.h5")
model.save(model_path)
print(f"Modelo recurrente guardado exitosamente en: {model_path}")

### Curvas de Entrenamiento y Comparativa

El modelo LSTM logra una precisión de validación del **60.13%** y un Log-loss de **0.8753**, lo que demuestra que las secuencias de partidos capturan patrones de racha significativos. Aunque el log-loss es ligeramente superior al baseline de regresión logística (`0.82`), el valor de la red recurrente radica en su capacidad de extraer representaciones secuenciales (embeddings), las cuales utilizaremos en nuestro modelo XGBoost integrado final.